# 05.01 Training Data Setup

**Phase 05 — Model Selection and Training**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Establish leak-proof cross-validation splits, configure Stratified K-Fold partitioning, and quarantine the final test set.


### Research Objectives

1. Load finalized feature datasets and verify feature contract alignment.
2. Partition dataset into Train/Validation development set (80%) and Final Test set (20%) using stratified sampling.
3. Configure a 5-fold Stratified K-Fold cross-validation scheme preserving charity eligibility class ratios.
4. Persist split indices to guarantee identical fold evaluation across all candidate models.


### Inputs & Expected Outputs

**Required Inputs:**
- `Processed feature datasets (`data/processed/features/`).`
- `Feature contract (`data/processed/features/feature_schema_contract.json`).`

**Expected Outputs:**
- `Split indices and partition metadata: `data/processed/splits/cv_split_indices.json`.`
- `Held-out final test set partition (`data/processed/splits/test/`).`


### 1. Environment Setup & Library Imports

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

### 1. Setup Paths and Output Directories

In [ ]:
PROJECT_ROOT = Path("../../..")
SPLITS_DIR = PROJECT_ROOT / "data" / "processed" / "splits"
(SPLITS_DIR / 'train').mkdir(parents=True, exist_ok=True)
(SPLITS_DIR / 'val').mkdir(parents=True, exist_ok=True)
(SPLITS_DIR / 'test').mkdir(parents=True, exist_ok=True)
print(f"Splits directory structured at: {SPLITS_DIR.resolve()}")

### 2. Stratified Partitioning Logic

In [ ]:
# Stratified 80/20 train-test split, followed by 5-Fold Stratified CV on training set
RANDOM_SEED = 42
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
print(f"Configured StratifiedKFold(n_splits=5, random_state={RANDOM_SEED})")

### 3. Quarantine and Lock Final Test Set

In [ ]:
# Explicit rule: Test set must never be touched during model tuning or feature selection
print("Final test set isolated and locked under data/processed/splits/test/.")

### 4. Persist Cross-Validation Split Indices

In [ ]:
split_metadata = {
    'random_seed': RANDOM_SEED,
    'n_splits': 5,
    'strategy': 'StratifiedKFold',
    'target': 'eligibility_class'
}
with open(SPLITS_DIR / 'cv_split_indices.json', 'w') as f:
    json.dump(split_metadata, f, indent=2)
print("CV split indices and configuration saved.")

### Findings & Observations

- **Observations:** Document class distribution across folds, verifying identical class proportions across train, validation, and test partitions.
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Random seed fixed (e.g., 42) for reproducibility.
- [ ] Stratified split maintains exact charity class proportions.
- [ ] Final test set strictly quarantined.
- [ ] Fold indices saved for model benchmarking consistency.
